In [ ]:
%pip install pyodbc

In [4]:
import pyodbc

# "Server=localhost\\SQLEXPRSS"
conn_str = (
    "Driver={ODBC Driver 18 for SQL Server};"
    "Server=localhost;"
    "Database=AdventureWorks2019;"
    "UID=python;"
    "PWD=python;"
    "TrustServerCertificate=Yes;"
)

try:
    conn = pyodbc.connect(conn_str)
    conn.close()
except Exception as e:
    print(e)

# cursor: ci permette di inviare SQL attraverso una conn

In [ ]:
import pyodbc

# "Server=localhost\\SQLEXPRSS"
conn_str = (
    "Driver={ODBC Driver 18 for SQL Server};"
    "Server=localhost;"
    "Database=AdventureWorks2019;"
    "UID=python;"
    "PWD=python;"
    "TrustServerCertificate=Yes;"
)

try:
    conn = pyodbc.connect(conn_str)

    cursor = conn.cursor()
    cursor.execute("SELECT TOP 50 * FROM Sales.Customer")

    for row in cursor.fetchall():
        print(row)

    conn.close()
except Exception as e:
    print(e)

# Manipolazione dei dati SQL con Pandas

In [ ]:
%pip install pandas

In [ ]:
import pyodbc
import pandas as pd

# "Server=localhost\\SQLEXPRSS"
conn_str = (
    "Driver={ODBC Driver 18 for SQL Server};"
    "Server=localhost;"
    "Database=AdventureWorks2019;"
    "UID=python;"
    "PWD=python;"
    "TrustServerCertificate=Yes;"
)

try:
    conn = pyodbc.connect(conn_str)

    query = "SELECT TOP 50 * FROM Sales.Customer"
    df = pd.read_sql(query, conn)

    print(df)

    conn.close()
except Exception as e:
    print(e)

In [ ]:
%pip install SQLAlchemy

# opzionalmente se da errore l'installazione di SQLAlchemy per problema alla wheel greenlet
%pip install --only-binary :all: greenlet

In [ ]:
import sqlalchemy as sa
import pandas as pd

# "Server=localhost\\SQLEXPRSS"
conn_str = (
    "Driver={ODBC Driver 18 for SQL Server};"
    "Server=localhost;"
    "Database=AdventureWorks2019;"
    "UID=python;"
    "PWD=python;"
    "TrustServerCertificate=Yes;"
)

try:
    connection_url = sa.URL.create("mssql+pyodbc", query={"odbc_connect": conn_str})
    engine = sa.create_engine(connection_url)

    with engine.begin() as conn: # serve per fare auto-close di conn alla fine del blocco
        query = "SELECT TOP 50 * FROM Sales.Customer"
        df = pd.read_sql_query(query, conn)
        df.info()
        print(df)

except Exception as e:
    print(e)

In [ ]:
import sqlalchemy as sa
import numpy as np
import pandas as pd

# "Server=localhost\\SQLEXPRSS"
conn_str = (
    "Driver={ODBC Driver 18 for SQL Server};"
    "Server=localhost;"
    "Database=AdventureWorks2019;"
    "UID=python;"
    "PWD=python;"
    "TrustServerCertificate=Yes;"
)

try:
    connection_url = sa.URL.create("mssql+pyodbc", query={"odbc_connect": conn_str})
    engine = sa.create_engine(connection_url)

    with engine.begin() as conn: # serve per fare auto-close di conn alla fine del blocco
        query = "SELECT * FROM [Production].[Product]"
        df = pd.read_sql_query(query, conn)
        # df.info()

        calcoli = df.agg(
            sumStock = ('SafetyStockLevel', 'sum'),
            avgStock = ('SafetyStockLevel', np.mean)
        )
        print(calcoli)

except Exception as e:
    print(e)

In [ ]:
import sqlalchemy as sa
import numpy as np
import pandas as pd

# "Server=localhost\\SQLEXPRSS"
conn_str = (
    "Driver={ODBC Driver 18 for SQL Server};"
    "Server=localhost;"
    "Database=AdventureWorks2019;"
    "UID=python;"
    "PWD=python;"
    "TrustServerCertificate=Yes;"
)

try:
    connection_url = sa.URL.create("mssql+pyodbc", query={"odbc_connect": conn_str})
    engine = sa.create_engine(connection_url)

    with engine.begin() as conn: # serve per fare auto-close di conn alla fine del blocco
        color = 'Black'
        # query = f"SELECT * FROM [Production].[Product] WHERE Color = '{color}'"
        query = sa.text("SELECT * FROM [Production].[Product] WHERE Color = :color") # query parametrica

        df = pd.read_sql(query, conn, params={ 'color': color })
        #df.info()

        df = df.dropna(subset=["Size"])
        df['AnnoModifica'] = df['ModifiedDate'].dt.year
        df.info()
        df.to_csv('ordini.csv')


except Exception as e:
    print(e)

In [62]:
import sqlalchemy as sa
import pandas as pd

# "Server=localhost\\SQLEXPRSS"
conn_str = (
    "Driver={ODBC Driver 18 for SQL Server};"
    "Server=localhost;"
    "Database=AdventureWorks2019;"
    "UID=python;"
    "PWD=python;"
    "TrustServerCertificate=Yes;"
)

try:
    connection_url = sa.URL.create("mssql+pyodbc", query={"odbc_connect": conn_str})
    engine = sa.create_engine(connection_url)

    df = pd.read_csv("..\\titanic\\train.csv")    

    with engine.begin() as conn: # serve per fare auto-close di conn alla fine del blocco
        # DDL per creare la tabella dei passeggeri
        # conn.execute(sa.text("""CREATE TABLE Passeggeri
        #             (
        #                 PassengerId int NOT NULL,
        #                 Survived bit NOT NULL,
        #                 Name varchar(100) NOT NULL
        #             )"""))
        # conn.execute(sa.text("""ALTER TABLE Passeggeri ADD CONSTRAINT
        #                 PK_Passeggeri PRIMARY KEY CLUSTERED 
        #                 (
        #                     PassengerId
        #                 )"""))
        for _, row in df.iterrows():
            conn.execute(sa.text("""
                INSERT INTO [dbo].[Passeggeri]
                    ([PassengerId]
                    ,[Survived]
                    ,[Name])
                VALUES
                    (:id 
                    ,:survived
                    ,:name)                
                """), {
                    "id": int(row["PassengerId"]),
                    "survived": int(row["Survived"]),
                    "name": row["Name"]
                })

except Exception as e:
    print(e)

In [ ]:
import sqlalchemy as sa
import pandas as pd

# "Server=localhost\\SQLEXPRSS"
conn_str = (
    "Driver={ODBC Driver 18 for SQL Server};"
    "Server=localhost;"
    "Database=AdventureWorks2019;"
    "UID=python;"
    "PWD=python;"
    "TrustServerCertificate=Yes;"
)

try:
    connection_url = sa.URL.create("mssql+pyodbc", query={"odbc_connect": conn_str})
    engine = sa.create_engine(connection_url)

    df = pd.read_csv("..\\titanic\\train.csv")    
    
    df.to_sql(
        "Titanic",
        engine,
        if_exists="replace",
        index=False
    )

except Exception as e:
    print(e)